# Resume multiple seeds of the physical-parameterization UIEB ablation

Attach the complete output directory from a stopped Kaggle job, enable Internet and a GPU, select one or more seeds, and run all cells. Do not point multiple seeds at one checkpoint file. Instead, the runner restores the complete output tree and resolves each configuration/seed's own `last_model.pth` independently (falling back to `best_model.pth` only for older outputs).

Completed runs are skipped, incomplete runs resume at the next epoch, and missing runs start from scratch.

In [ ]:
from pathlib import Path

# Any non-empty, duplicate-free subset is valid. The job processes these sequentially.
SEEDS_TO_RUN = [0, 1, 2]

# Point to the restored output root, not to one seed's checkpoint. It should contain
# directories such as physical_reconstruction/seed_0/last_model.pth.
PREVIOUS_OUTPUT_ROOT = Path(
    '/kaggle/input/REPLACE_WITH_PREVIOUS_OUTPUT/physics_parameterization_ablation_uieb'
)
assert SEEDS_TO_RUN, 'Select at least one seed'
assert len(SEEDS_TO_RUN) == len(set(SEEDS_TO_RUN)), f'Duplicate seeds: {SEEDS_TO_RUN}'
assert all(isinstance(seed, int) and seed >= 0 for seed in SEEDS_TO_RUN), SEEDS_TO_RUN
assert PREVIOUS_OUTPUT_ROOT.is_dir(), (
    f'Edit PREVIOUS_OUTPUT_ROOT; directory not found: {PREVIOUS_OUTPUT_ROOT}'
)
print('Seeds requested:', SEEDS_TO_RUN)
print('Previous output root:', PREVIOUS_OUTPUT_ROOT)

In [ ]:
import os
import subprocess

REPO = Path('/kaggle/working/underwater-image-enhancement')
BRANCH = 'learnable-physics-extractor'
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch', '--depth', '1',
        'https://github.com/heniath/underwater-image-enhancement.git', str(REPO),
    ], check=True)
# The main runner copies this complete tree into /kaggle/working and then
# selects the matching checkpoint separately for every configuration/seed.
os.environ.pop('UWIR_RESUME_CHECKPOINT', None)
os.environ['UWIR_PREVIOUS_OUTPUT_ROOT'] = str(PREVIOUS_OUTPUT_ROOT)
os.environ['UWIR_SEEDS_TO_RUN'] = ','.join(map(str, SEEDS_TO_RUN))
os.chdir(REPO)
runner_candidates = [
    REPO / 'learnable_physics_uieb.ipynb',
    REPO / 'learnable_physics_uieb_kaggle.ipynb',  # backward compatibility
]
runner = next((path for path in runner_candidates if path.is_file()), None)
assert runner is not None, f'Runner notebook not found on {BRANCH}: {runner_candidates}'
get_ipython().run_line_magic('run', f'-i {runner}')